In [1]:
# 函数调用
import os
api_key = os.environ.get("DEEPSEEK_API_KEY")

In [2]:
# 定义一个获取当前天气的函数
import json

def get_current_weather(location, unit="fahrenheit"):
    """Get the current weather in given location"""
    weather_info = {
        "location" : location,
        "temperature" : "72",
        "unit" : unit,
        "forecast" : ["sunny", "windy"],
    }
    return json.dumps(weather_info)
    # 目前通过硬编码返回信息

In [3]:
# 传递消息给语言模型，OpenAI新增了一个名为function说的参数 ，拆的你一组函数定义
# 其中包含几个不同参数的JSON对象
functions = [
    {
        "name": "get_current_weather",
        "description": "Get the current weather in a given location",
        "parameters": {
            "type": "object",
            "properties": {
                "location": {
                    "type": "string",
                    "description": "The city and state, e.g. San Francisco, CA",
                },
                "unit": {"type": "string", "enum": ["celsius", "fahrenheit"]},
            },
            "required": ["location"],
        },
    }
]

In [4]:
# 创建消息列表
messages = [
    {
        "role": "user",
        "content" : "What's the weather like in Boston?"
    }
]

In [5]:
from openai import OpenAI

In [6]:
# 此处为1.0版本以上的openai包，因此需要创建client对象
client = OpenAI(
    base_url="https://api.deepseek.com/v1", # 本地的deepseek-v4-flash服务url格式
    api_key=api_key # deepseek-v4-flash不需要api_key
)

In [7]:
response = client.chat.completions.create(
    # 需要启动本地deepseek-v4-flash服务，可在任务管理器当中打开默认启动
    model="deepseek-v4-flash",
    messages=messages,
    tools=[{"type":"function","function":functions[0]}]
)

In [8]:
print(response)

ChatCompletion(id='6f5b64bb-6702-486d-8fc6-654b8ae9a182', choices=[Choice(finish_reason='tool_calls', index=0, logprobs=None, message=ChatCompletionMessage(content='', refusal=None, role='assistant', annotations=None, audio=None, function_call=None, tool_calls=[ChatCompletionMessageFunctionToolCall(id='call_00_71SRRGl8HeiaM9AlvBji0368', function=Function(arguments='{"location": "Boston, MA", "unit": "fahrenheit"}', name='get_current_weather'), type='function', index=0)], reasoning_content='The user asks about weather in Boston. I need to get current weather. Unit not specified - default to fahrenheit probably? I can ask or choose fahrenheit. Let me call the tool.'))], created=1788787178, model='deepseek-v4-flash', object='chat.completion', metadata=None, moderation=None, service_tier=None, system_fingerprint='a26a7955944dc5c60445bff77fac9c8e', usage=CompletionUsage(completion_tokens=104, prompt_tokens=400, total_tokens=504, completion_tokens_details=CompletionTokensDetails(accepted_pre

In [9]:
response_reasoning = response.choices[0].message # 打印message输出
# 查看返回的消息
response_reasoning

ChatCompletionMessage(content='', refusal=None, role='assistant', annotations=None, audio=None, function_call=None, tool_calls=[ChatCompletionMessageFunctionToolCall(id='call_00_71SRRGl8HeiaM9AlvBji0368', function=Function(arguments='{"location": "Boston, MA", "unit": "fahrenheit"}', name='get_current_weather'), type='function', index=0)], reasoning_content='The user asks about weather in Boston. I need to get current weather. Unit not specified - default to fahrenheit probably? I can ask or choose fahrenheit. Let me call the tool.')

In [10]:
response_reasoning.reasoning_content

'The user asks about weather in Boston. I need to get current weather. Unit not specified - default to fahrenheit probably? I can ask or choose fahrenheit. Let me call the tool.'

In [11]:
response_reasoning.tool_calls

[ChatCompletionMessageFunctionToolCall(id='call_00_71SRRGl8HeiaM9AlvBji0368', function=Function(arguments='{"location": "Boston, MA", "unit": "fahrenheit"}', name='get_current_weather'), type='function', index=0)]

In [12]:
response_reasoning.tool_calls[0].function.arguments

'{"location": "Boston, MA", "unit": "fahrenheit"}'

In [13]:
json.loads(response_reasoning.tool_calls[0].function.arguments)

{'location': 'Boston, MA', 'unit': 'fahrenheit'}

In [14]:
args = json.loads(response_reasoning.tool_calls[0].function.arguments)

In [15]:
# OpenAI的函数调用并不会直接让大模型调用函数，而是告知应该调用哪个函数，包括函数名称和参数
get_current_weather(args)

'{"location": {"location": "Boston, MA", "unit": "fahrenheit"}, "temperature": "72", "unit": "fahrenheit", "forecast": ["sunny", "windy"]}'

In [16]:
# 查看与输入无关的测试结果
messages = [
    {
        "role": "user",
        "content": "hi",
    }
]

In [17]:
messages

[{'role': 'user', 'content': 'hi'}]

In [18]:
# 重新创建客户端对象
client = OpenAI(
    base_url="https://api.deepseek.com/v1",
    api_key=api_key
)

In [19]:
response = client.chat.completions.create(
    model="deepseek-v4-flash",
    messages=messages,
    functions=functions,
    # 关闭思考模式
    extra_body={
        "think":False
    },
)

In [20]:
print(response.choices[0].message.tool_calls) # 可以看到此处function_call为空

None


In [21]:
print(response.choices[0].message.content)

Hi! How can I help you today?


In [22]:
# 通过传递额外参数强制模型去使用或者不使用函数
response = client.chat.completions.create(
    model="deepseek-v4-flash",
    messages=messages,
    tools=[{"type": "function", "function": f} for f in functions],
    # 关闭思考模式
    reasoning_effort="none",
    tool_choice="auto" # llm会自动选择
)

In [23]:
response

ChatCompletion(id='0577c615-0608-43a3-b0af-0963e971cec2', choices=[Choice(finish_reason='stop', index=0, logprobs=None, message=ChatCompletionMessage(content='Hello! How can I help you today?', refusal=None, role='assistant', annotations=None, audio=None, function_call=None, tool_calls=None))], created=1788787181, model='deepseek-v4-flash', object='chat.completion', metadata=None, moderation=None, service_tier=None, system_fingerprint='a26a7955944dc5c60445bff77fac9c8e', usage=CompletionUsage(completion_tokens=9, prompt_tokens=315, total_tokens=324, completion_tokens_details=None, prompt_tokens_details=PromptTokensDetails(audio_tokens=None, cache_write_tokens=None, cached_tokens=256, image_tokens=None, text_tokens=None), prompt_cache_hit_tokens=256, prompt_cache_miss_tokens=59))

In [24]:
messages = [
    {
        "role":"user",
        "content":"What's the weather in Boston"
    }
]

response = client.chat.completions.create(
    model="deepseek-v4-flash",
    messages=messages,
    tools=[{"type":"function","function":f} for f in functions],
    # 关闭思考模式
    reasoning_effort="none",
    tool_choice="none"
)



In [25]:
print(response)

ChatCompletion(id='a204ffdf-7dea-4ce1-a807-47ac11440c29', choices=[Choice(finish_reason='stop', index=0, logprobs=None, message=ChatCompletionMessage(content="I don't have real-time access to current weather data, so I can't tell you the exact conditions in Boston right now. \n\nTo get the most accurate and up-to-date forecast, I recommend checking a reliable weather service like:\n\n- **Weather.com** (The Weather Channel)\n- **AccuWeather**\n- **The National Weather Service** (weather.gov)\n- Or asking your phone's built-in virtual assistant (Siri, Google Assistant, etc.)\n\n**However**, if you are asking about the *typical* weather for today's date (May 16th), Boston generally has mild spring weather. Average highs are around the upper 60s°F (about 20°C), with overnight lows in the upper 40s°F (about 8-9°C). It's often a mix of sun and clouds, and there's always a chance of a spring shower.\n\nIf you'd like, you can tell me your location or ask me how to check the weather on your spe

In [26]:
messages = [
    {
        "role":"user",
        "content":"What's the weather in Boston"
    }
]
# 强制调用函数
response = client.chat.completions.create(
    model="deepseek-v4-flash",
    messages=messages,
    tools=[
        {
            "type":"function",
            "function":f,
        }   for f in functions
    ],
    # 关闭思考模式
    reasoning_effort="none",
    # 新版OpenAI已经废弃了function_call，现在设置函数调用改为tool_choice
    # 同时functions也需要改为tools
    tool_choice={
        "type":"function",
        "function":{
            "name":"get_current_weather"
        },
    }
)

print(response)

ChatCompletion(id='b2838715-829c-4956-84bc-a07931b90a12', choices=[Choice(finish_reason='tool_calls', index=0, logprobs=None, message=ChatCompletionMessage(content='', refusal=None, role='assistant', annotations=None, audio=None, function_call=None, tool_calls=[ChatCompletionMessageFunctionToolCall(id='call_00_8PVI0lD7EEunW5VrRlTu1868', function=Function(arguments='{"location": "Boston"}', name='get_current_weather'), type='function', index=0)]))], created=1788787185, model='deepseek-v4-flash', object='chat.completion', metadata=None, moderation=None, service_tier=None, system_fingerprint='a26a7955944dc5c60445bff77fac9c8e', usage=CompletionUsage(completion_tokens=38, prompt_tokens=327, total_tokens=365, completion_tokens_details=None, prompt_tokens_details=PromptTokensDetails(audio_tokens=None, cache_write_tokens=None, cached_tokens=256, image_tokens=None, text_tokens=None), prompt_cache_hit_tokens=256, prompt_cache_miss_tokens=71))


In [27]:
response.choices[0].message.tool_calls

[ChatCompletionMessageFunctionToolCall(id='call_00_8PVI0lD7EEunW5VrRlTu1868', function=Function(arguments='{"location": "Boston"}', name='get_current_weather'), type='function', index=0)]

In [28]:
messages = [
    {
        "role":"user",
        "content":"What's the weather like in Boston?", # 当不需要调用函数时，强制调用函数
    }
]

# 强制调用函数
response = client.chat.completions.create(
    model="deepseek-v4-flash",
    messages=messages,
    tools=[
        {
            "type":"function",
            "function": f,
        } for f in functions
    ],
    reasoning_effort="none",
    tool_choice={
        "type":"function",
        "function":{
            "name":"get_current_weather",
        }
    },
)

# 函数本身机器描述会占用传递给OpenAI的令牌使用限制，当注释掉函数调用，提示令牌降至18
response

ChatCompletion(id='4b2db518-490b-4806-89cd-f1ddef8579a6', choices=[Choice(finish_reason='tool_calls', index=0, logprobs=None, message=ChatCompletionMessage(content='', refusal=None, role='assistant', annotations=None, audio=None, function_call=None, tool_calls=[ChatCompletionMessageFunctionToolCall(id='call_00_Y01d4qpye1wutXfg50gn8172', function=Function(arguments='{"location": "Boston, MA", "unit": "fahrenheit"}', name='get_current_weather'), type='function', index=0)]))], created=1788787186, model='deepseek-v4-flash', object='chat.completion', metadata=None, moderation=None, service_tier=None, system_fingerprint='a26a7955944dc5c60445bff77fac9c8e', usage=CompletionUsage(completion_tokens=57, prompt_tokens=329, total_tokens=386, completion_tokens_details=None, prompt_tokens_details=PromptTokensDetails(audio_tokens=None, cache_write_tokens=None, cached_tokens=256, image_tokens=None, text_tokens=None), prompt_cache_hit_tokens=256, prompt_cache_miss_tokens=73))

In [29]:
print(response.usage)

CompletionUsage(completion_tokens=57, prompt_tokens=329, total_tokens=386, completion_tokens_details=None, prompt_tokens_details=PromptTokensDetails(audio_tokens=None, cache_write_tokens=None, cached_tokens=256, image_tokens=None, text_tokens=None), prompt_cache_hit_tokens=256, prompt_cache_miss_tokens=73)


In [30]:
response.choices[0].message

ChatCompletionMessage(content='', refusal=None, role='assistant', annotations=None, audio=None, function_call=None, tool_calls=[ChatCompletionMessageFunctionToolCall(id='call_00_Y01d4qpye1wutXfg50gn8172', function=Function(arguments='{"location": "Boston, MA", "unit": "fahrenheit"}', name='get_current_weather'), type='function', index=0)])

In [31]:
# 传递函数调用
messages.append(response.choices[0].message)

In [32]:
args = json.loads(response.choices[0].message.tool_calls[0].function.arguments) # 注意新版本与课程代码的区别

observation = get_current_weather(**args)


In [33]:
messages.append(
    {
        "role": "tool", # 新版本这里为tool，同时不是使用name指明，而是messsage当中的id
        "tool_call_id": response.choices[0].message.tool_calls[0].id,
        "content": observation,
    }
)

In [34]:
messages

[{'role': 'user', 'content': "What's the weather like in Boston?"},
 ChatCompletionMessage(content='', refusal=None, role='assistant', annotations=None, audio=None, function_call=None, tool_calls=[ChatCompletionMessageFunctionToolCall(id='call_00_Y01d4qpye1wutXfg50gn8172', function=Function(arguments='{"location": "Boston, MA", "unit": "fahrenheit"}', name='get_current_weather'), type='function', index=0)]),
 {'role': 'tool',
  'tool_call_id': 'call_00_Y01d4qpye1wutXfg50gn8172',
  'content': '{"location": "Boston, MA", "temperature": "72", "unit": "fahrenheit", "forecast": ["sunny", "windy"]}'}]

In [37]:
response = client.chat.completions.create(
    model="deepseek-v4-flash",
    messages=messages,
    reasoning_effort="none",
)

print(response.choices[0])

Choice(finish_reason='stop', index=0, logprobs=None, message=ChatCompletionMessage(content="Right now in Boston, the weather is **72°F (about 22°C)** and **sunny**, though it is also **windy**. \n\nIf you're heading out, you might want a light jacket since the wind could make it feel a bit cooler.", refusal=None, role='assistant', annotations=None, audio=None, function_call=None, tool_calls=None))


In [40]:
print(response.choices[0].message.content)

Right now in Boston, the weather is **72°F (about 22°C)** and **sunny**, though it is also **windy**. 

If you're heading out, you might want a light jacket since the wind could make it feel a bit cooler.
